In [ ]:
!pip install -q timm albumentations safetensors wandb huggingface_hub datasets scikit-learn scipy

In [ ]:
import subprocess, sys, pathlib, importlib

repo_dir = pathlib.Path("/kaggle/working/anemia-detection-ai")
if repo_dir.exists():
    result = subprocess.run(["git", "-C", str(repo_dir), "pull"], capture_output=True, text=True)
else:
    result = subprocess.run(
        ["git", "clone", "https://github.com/hssling/anemia-detection-ai.git", str(repo_dir)],
        capture_output=True, text=True
    )
print(result.stdout, result.stderr)

# Flush any cached training modules so re-runs pick up the latest code
for _mod in [k for k in sys.modules if k.startswith("training")]:
    del sys.modules[_mod]
importlib.invalidate_caches()

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    _hf_token = _s.get_secret("HF_TOKEN").strip()
    _wandb_key = _s.get_secret("WANDB_API_KEY").strip()
    print("Loaded secrets via UserSecretsClient")
except Exception as _e:
    print(f"kaggle_secrets unavailable ({_e}), falling back to env vars")
    _hf_token = os.environ.get("HF_TOKEN", "").strip()
    _wandb_key = os.environ.get("WANDB_API_KEY", "").strip()

if not _hf_token:
    raise RuntimeError(
        "HF_TOKEN is empty. In Kaggle: Edit -> Add-ons -> Secrets -> Add Secret (name: HF_TOKEN) and enable the toggle."
    )
if not _wandb_key:
    raise RuntimeError(
        "WANDB_API_KEY is empty. In Kaggle: Edit -> Add-ons -> Secrets -> Add Secret (name: WANDB_API_KEY) and enable the toggle."
    )

from huggingface_hub import login
login(token=_hf_token)
import wandb
os.environ["WANDB_API_KEY"] = _wandb_key
wandb.login(key=_wandb_key)
print("Auth complete")

In [ ]:
from datasets import load_dataset
dataset = load_dataset("hssling/anemia-conjunctiva-nailbed")
train_conj = [r for r in dataset["train"] if r["site"] == "conjunctiva"]
val_conj   = [r for r in dataset["val"]   if r["site"] == "conjunctiva"]
train_nail = [r for r in dataset["train"] if r["site"] == "nailbed"]
val_nail   = [r for r in dataset["val"]   if r["site"] == "nailbed"]
print(f"Train conj: {len(train_conj)}, val conj: {len(val_conj)}")
print(f"Train nail: {len(train_nail)}, val nail: {len(val_nail)}")

In [ ]:
import yaml, pathlib, os, sys, subprocess, importlib

# Always pull latest code and flush module cache before importing training modules
_repo = pathlib.Path("/kaggle/working/anemia-detection-ai")
_r = subprocess.run(["git", "-C", str(_repo), "pull"], capture_output=True, text=True)
print(_r.stdout.strip() or "git pull: already up to date", _r.stderr.strip() or "")
for _m in [k for k in sys.modules if k.startswith("training")]:
    del sys.modules[_m]
importlib.invalidate_caches()

os.chdir(str(_repo))
config = yaml.safe_load(open("training/config.yaml"))
from training.cross_validation import run_cross_validation
cv_summary_conj = run_cross_validation(
    rows=train_conj + val_conj,
    model_name="efficientnet_b4",
    config=config,
    output_dir=pathlib.Path("/kaggle/working/outputs/cv_conj"),
)
print("CV conjunctiva:", cv_summary_conj)

In [ ]:
from training.train import train_model
final_metrics_conj = train_model(
    model_name="efficientnet_b4",
    train_rows=train_conj + val_conj,
    val_rows=val_conj,
    config=config,
    output_dir=pathlib.Path("/kaggle/working/outputs/final"),
    fold=99,
    run_name="efficientnet_b4_conjunctiva_final",
)
print("Final conjunctiva metrics:", final_metrics_conj)

In [ ]:
cv_summary_nail = run_cross_validation(
    rows=train_nail + val_nail,
    model_name="efficientnet_b4",
    config=config,
    output_dir=pathlib.Path("/kaggle/working/outputs/cv_nail"),
)
print("CV nail-bed:", cv_summary_nail)

In [ ]:
final_metrics_nail = train_model(
    model_name="efficientnet_b4",
    train_rows=train_nail + val_nail,
    val_rows=val_nail,
    config=config,
    output_dir=pathlib.Path("/kaggle/working/outputs/final"),
    fold=98,
    run_name="efficientnet_b4_nailbed_final",
)
print("Final nail-bed metrics:", final_metrics_nail)

In [ ]:
from training.models.ensemble import AnemiaEnsemble
w_conj, w_nail = AnemiaEnsemble.find_best_weights(
    conj_ckpt="/kaggle/working/outputs/final/efficientnet_b4_fold99_best.safetensors",
    nail_ckpt="/kaggle/working/outputs/final/efficientnet_b4_fold98_best.safetensors",
    val_rows_conj=val_conj,
    val_rows_nail=val_nail,
    config=config,
)
print(f"Best ensemble weights: w_conj={w_conj:.2f}, w_nail={w_nail:.2f}")

In [ ]:
from training.push_model_to_hf import push_all_models
push_all_models(
    conj_ckpt="/kaggle/working/outputs/final/efficientnet_b4_fold99_best.safetensors",
    nail_ckpt="/kaggle/working/outputs/final/efficientnet_b4_fold98_best.safetensors",
    cv_summary_conj=cv_summary_conj,
    cv_summary_nail=cv_summary_nail,
    w_conj=w_conj,
    w_nail=w_nail,
    config=config,
)
print("All models pushed to HuggingFace Hub")